# Day 12 v3 — Gold Dimensional: Production Job Params

**Production notebook — ADF/Databricks Jobs sends only `load_type`.**
All date params auto-compute (yesterday UTC) with Python defaults.

Runs the full dimensional Gold pipeline in order:
1. **Dims** — 13 dimension tables (full overwrite each run; small, fast)
2. **Facts** — 7 fact tables (full or Delta MERGE depending on `load_type`)
3. **Marts** — 9 aggregated marts (full overwrite each run; derived from facts)

**ADF `baseParameters`:** only `load_type=incremental` (or `full` for backfill).

**Incremental fact MERGE keys:**

| Fact | MERGE key |
|---|---|
| FactChargingSession | `session_id` |
| FactEnergyConsumption | `charger_key`, `session_hour_ts` |
| FactPayments | `payment_id` |
| FactMaintenance | `event_id` |
| FactDeviceTelemetry | `session_id` |
| FactComplaints | `complaint_id` |
| FactStationUtilisation | `station_key`, `session_hour_ts` |

In [ ]:
# Cell 1 — Imports
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime, timezone, timedelta

print('Imports OK')

In [ ]:
# Cell 2 — Job params (ADF passes load_type only; all date/time defaults auto-computed)
_now      = datetime.now(timezone.utc)
_yesterday = _now - timedelta(days=1)

def _get_param(key, default):
    try:
        val = dbutils.widgets.get(key).strip()
        return val if val else default
    except Exception:
        return default

LOAD_TYPE = _get_param('load_type', 'incremental')
RUN_DATE  = _get_param('run_date',  _yesterday.strftime('%Y-%m-%d'))  # default: yesterday UTC
RUN_TS    = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

print(f'LOAD_TYPE : {LOAD_TYPE}')
print(f'RUN_DATE  : {RUN_DATE}')
print(f'RUN_TS    : {RUN_TS}')

In [ ]:
# Cell 3 — Constants
SILVER_API = '/Volumes/dbw_ev_intelligence_dev/default/silver-volume/api'
SILVER_RT  = '/Volumes/dbw_ev_intelligence_dev/default/silver-volume/realtime'
GOLD_DIM   = '/Volumes/dbw_ev_intelligence_dev/default/gold-volume/dimensional/dims'
GOLD_FACT  = '/Volumes/dbw_ev_intelligence_dev/default/gold-volume/dimensional/facts'
GOLD_MART  = '/Volumes/dbw_ev_intelligence_dev/default/gold-volume/dimensional/marts'
PIPELINE   = f'pl_gold_dimensional_v3_{LOAD_TYPE}'

def read_silver(name):
    return spark.read.format('delta').load(f'{SILVER_API}/{name}')

def read_dim(name):
    return spark.read.format('delta').load(f'{GOLD_DIM}/{name}')

def read_fact(name):
    return spark.read.format('delta').load(f'{GOLD_FACT}/{name}')

# write_gold: full overwrite for dims/marts, Delta MERGE for facts on incremental
def write_gold(df, path, merge_keys, partition_cols=None):
    if LOAD_TYPE == 'incremental' and merge_keys:
        if DeltaTable.isDeltaTable(spark, path):
            dt = DeltaTable.forPath(spark, path)
            cond = ' AND '.join([f'tgt.{k} = src.{k}' for k in merge_keys])
            dt.alias('tgt').merge(df.alias('src'), cond) \
              .whenMatchedUpdateAll() \
              .whenNotMatchedInsertAll() \
              .execute()
            return
    # Full overwrite (new table or full load)
    writer = df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true')
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)
    writer.save(path)

def write_dim(df, dim_name):
    path = f'{GOLD_DIM}/{dim_name}'
    df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').save(path)
    print(f'  [DIM]  {dim_name:<25} {df.count():>7} rows')

def write_fact(df, fact_name, merge_keys, partition_cols=None):
    path = f'{GOLD_FACT}/{fact_name}'
    write_gold(df, path, merge_keys, partition_cols)
    cnt = spark.read.format('delta').load(path).count()
    mode = 'MERGE' if LOAD_TYPE == 'incremental' and DeltaTable.isDeltaTable(spark, path) else 'OVERWRITE'
    print(f'  [FACT] {fact_name:<30} {cnt:>7} rows ({mode})')

def write_mart(df, mart_name, partition_cols=None):
    path = f'{GOLD_MART}/{mart_name}'
    writer = df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true')
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)
    writer.save(path)
    print(f'  [MART] {mart_name:<35} {df.count():>7} rows')

def time_key_expr(ts_col):
    return F.concat_ws('',
        F.year(ts_col).cast('string'),
        F.lpad(F.month(ts_col).cast('string'),      2, '0'),
        F.lpad(F.dayofmonth(ts_col).cast('string'), 2, '0'),
        F.lpad(F.hour(ts_col).cast('string'),       2, '0'),
    )

print(f'Pipeline  : {PIPELINE}')
print(f'Gold paths: dims={GOLD_DIM} | facts={GOLD_FACT} | marts={GOLD_MART}')

In [ ]:
# Cell 4 — Read Silver sources (once; reused by dims + facts)
states_raw    = read_silver('states')
cities_raw    = read_silver('cities')
stations_raw  = read_silver('stations')
chargers_raw  = read_silver('chargers')
customers_raw = read_silver('customers')
vehicles_raw  = read_silver('vehicles')
employees_raw = read_silver('employees')
partners_raw  = read_silver('partners')
cards_raw     = read_silver('charge_cards')
tariffs_raw   = read_silver('tariffs')
weather_raw   = read_silver('weather')
sessions_raw  = read_silver('sessions')
payments_raw  = read_silver('payments')
maintenance_r = read_silver('maintenance_events')
complaints_r  = read_silver('complaints')
rt_sessions   = spark.read.format('delta').load(f'{SILVER_RT}/charging_sessions')

# For incremental: filter facts to RUN_DATE only
if LOAD_TYPE == 'incremental':
    run_date_lit = F.to_date(F.lit(RUN_DATE))
    sessions_raw  = sessions_raw.filter(F.to_date('start_time')    == run_date_lit)
    payments_raw  = payments_raw.filter(F.to_date('payment_time')  == run_date_lit)
    maintenance_r = maintenance_r.filter(F.to_date('created_at')   == run_date_lit)
    complaints_r  = complaints_r.filter(F.to_date('created_at')    == run_date_lit)
    rt_sessions   = rt_sessions.filter(F.to_date('plug_in_ts')     == run_date_lit)
    print(f'Incremental mode — filtering facts to RUN_DATE={RUN_DATE}')
else:
    print('Full load — processing all Silver data')

print('Silver sources loaded')

In [ ]:
# Cell 5 — DIMS: build all 13 and write (always full overwrite)
print('=== DIMS ===')

# DimCountry
dim_country = (
    states_raw.select(F.trim(F.col('country')).alias('country_name')).distinct()
    .filter(F.col('country_name').isNotNull() & (F.col('country_name') != ''))
    .withColumn('country_key', F.monotonically_increasing_id())
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('country_key', 'country_name', 'gold_ingested_at', 'gold_pipeline')
)
write_dim(dim_country, 'DimCountry')

# DimState
dim_state = (
    states_raw.select(
        F.trim('state_code').alias('state_code'), F.trim('name').alias('state_name'), F.trim('country').alias('country_name')
    ).distinct().filter(F.col('state_code').isNotNull())
    .withColumn('state_key', F.monotonically_increasing_id())
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('state_key', 'state_code', 'state_name', 'country_name', 'gold_ingested_at', 'gold_pipeline')
)
write_dim(dim_state, 'DimState')

# DimCity
dim_city = (
    cities_raw.select(
        'city_id', F.trim('name').alias('city_name'), F.trim('state_code').alias('state_code'),
        F.trim('country').alias('country_name'), F.col('latitude').alias('city_lat'),
        F.col('longitude').alias('city_lon'), 'population'
    ).filter(F.col('city_id').isNotNull())
    .withColumn('city_key', F.monotonically_increasing_id())
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('city_key', 'city_id', 'city_name', 'state_code', 'country_name',
            'city_lat', 'city_lon', 'population', 'gold_ingested_at', 'gold_pipeline')
)
write_dim(dim_city, 'DimCity')

# DimStation
dim_station = (
    stations_raw.select(
        'station_id', F.trim('name').alias('station_name'), F.trim('city').alias('city_name'),
        F.trim('state').alias('state_code'), F.trim('country').alias('country_name'),
        F.col('latitude').alias('lat'), F.col('longitude').alias('lng'),
        'total_chargers', F.trim('status').alias('status')
    ).filter(F.col('station_id').isNotNull())
    .withColumn('station_key', F.monotonically_increasing_id())
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('station_key', 'station_id', 'station_name', 'city_name',
            'state_code', 'country_name', 'lat', 'lng',
            'total_chargers', 'status', 'gold_ingested_at', 'gold_pipeline')
)
write_dim(dim_station, 'DimStation')

# DimCharger (SCD2)
fw_win    = Window.partitionBy('charger_id').orderBy(F.col('plug_in_ts').desc())
latest_fw = (
    rt_sessions.filter(F.col('firmware_ver').isNotNull())
    .withColumn('_rn', F.row_number().over(fw_win)).filter(F.col('_rn') == 1).drop('_rn')
    .select('charger_id', F.col('firmware_ver').alias('firmware_version'))
)
dim_charger = (
    chargers_raw.join(latest_fw, on='charger_id', how='left')
    .select(
        'charger_id', 'station_id', F.trim('charger_type').alias('charger_type'),
        F.trim('connector_type').alias('connector_type'),
        F.col('power_kw').alias('max_kw'), F.trim('status').alias('status'),
        'firmware_version', F.col('created_at').alias('valid_from'),
        F.lit(None).cast('timestamp').alias('valid_to'), F.lit(True).alias('is_current')
    ).filter(F.col('charger_id').isNotNull())
    .withColumn('charger_key', F.monotonically_increasing_id())
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('charger_key', 'charger_id', 'station_id', 'charger_type', 'connector_type',
            'max_kw', 'status', 'firmware_version', 'valid_from', 'valid_to', 'is_current',
            'gold_ingested_at', 'gold_pipeline')
)
write_dim(dim_charger, 'DimCharger')

# DimCustomer (SCD2)
dim_customer = (
    customers_raw.select(
        'customer_id', F.trim('first_name').alias('first_name'), F.trim('last_name').alias('last_name'),
        F.concat_ws(' ', F.trim('first_name'), F.trim('last_name')).alias('full_name'),
        F.trim('email').alias('email'), F.trim('phone').alias('phone'),
        F.trim('city').alias('city'), F.trim('state').alias('state'), F.trim('country').alias('country'),
        F.to_date('created_at').alias('signup_date'),
        F.col('created_at').alias('valid_from'),
        F.lit(None).cast('timestamp').alias('valid_to'), F.lit(True).alias('is_current')
    ).filter(F.col('customer_id').isNotNull())
    .withColumn('customer_key', F.monotonically_increasing_id())
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('customer_key', 'customer_id', 'first_name', 'last_name', 'full_name',
            'email', 'phone', 'city', 'state', 'country', 'signup_date',
            'valid_from', 'valid_to', 'is_current', 'gold_ingested_at', 'gold_pipeline')
)
write_dim(dim_customer, 'DimCustomer')

# DimVehicle
dim_vehicle = (
    vehicles_raw.select(
        'vehicle_id', 'customer_id', F.trim('make').alias('make'), F.trim('model').alias('model'),
        'year', F.col('battery_capacity').alias('battery_capacity_kwh'),
        'range_km', F.trim('license_plate').alias('license_plate')
    ).filter(F.col('vehicle_id').isNotNull())
    .withColumn('vehicle_key', F.monotonically_increasing_id())
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('vehicle_key', 'vehicle_id', 'customer_id', 'make', 'model',
            'year', 'battery_capacity_kwh', 'range_km', 'license_plate',
            'gold_ingested_at', 'gold_pipeline')
)
write_dim(dim_vehicle, 'DimVehicle')

# DimEmployee
dim_employee = (
    employees_raw.select(
        'employee_id', F.trim('first_name').alias('first_name'), F.trim('last_name').alias('last_name'),
        F.concat_ws(' ', F.trim('first_name'), F.trim('last_name')).alias('full_name'),
        F.trim('email').alias('email'), F.trim('role').alias('role'),
        F.trim('department').alias('department'), 'station_id', 'hire_date'
    ).filter(F.col('employee_id').isNotNull())
    .withColumn('employee_key', F.monotonically_increasing_id())
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('employee_key', 'employee_id', 'first_name', 'last_name', 'full_name',
            'email', 'role', 'department', 'station_id', 'hire_date',
            'gold_ingested_at', 'gold_pipeline')
)
write_dim(dim_employee, 'DimEmployee')

# DimFranchisePartner
dim_partner = (
    partners_raw.select(
        'partner_id', F.trim('name').alias('partner_name'), F.trim('country').alias('country'),
        F.trim('contact_email').alias('contact_email'), F.trim('status').alias('status')
    ).filter(F.col('partner_id').isNotNull())
    .withColumn('partner_key', F.monotonically_increasing_id())
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('partner_key', 'partner_id', 'partner_name', 'country',
            'contact_email', 'status', 'gold_ingested_at', 'gold_pipeline')
)
write_dim(dim_partner, 'DimFranchisePartner')

# DimChargeCard
dim_card = (
    cards_raw.select(
        'card_id', 'customer_id',
        F.when(
            F.col('card_number').isNotNull() & (F.length('card_number') >= 8),
            F.concat(F.substring('card_number', 1, 4), F.lit('-****-****-'), F.substring('card_number', -4, 4))
        ).otherwise(F.lit('****')).alias('card_number_masked'),
        F.trim('status').alias('status'), 'issued_at', 'expires_at'
    ).filter(F.col('card_id').isNotNull())
    .withColumn('card_key', F.monotonically_increasing_id())
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('card_key', 'card_id', 'customer_id', 'card_number_masked',
            'status', 'issued_at', 'expires_at', 'gold_ingested_at', 'gold_pipeline')
)
write_dim(dim_card, 'DimChargeCard')

# DimTariff (SCD2)
dim_tariff = (
    tariffs_raw.select(
        'tariff_id', F.trim('name').alias('tariff_name'),
        F.col('price_per_kwh').alias('rate_per_kwh'), F.col('price_per_min').alias('rate_per_min'),
        F.trim('currency').alias('currency'),
        F.col('valid_from').alias('effective_from'), F.col('valid_to').alias('effective_to'),
        F.when(
            F.col('valid_to').isNull() | (F.col('valid_to') > F.lit(RUN_TS).cast('timestamp')),
            F.lit(True)
        ).otherwise(F.lit(False)).alias('is_current')
    ).filter(F.col('tariff_id').isNotNull())
    .withColumn('tariff_key', F.monotonically_increasing_id())
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('tariff_key', 'tariff_id', 'tariff_name', 'rate_per_kwh', 'rate_per_min',
            'currency', 'effective_from', 'effective_to', 'is_current',
            'gold_ingested_at', 'gold_pipeline')
)
write_dim(dim_tariff, 'DimTariff')

# DimWeather
city_lu = cities_raw.select('city_id', F.trim('name').alias('city_name'), F.trim('state_code').alias('state_code'))
dim_weather = (
    weather_raw.join(city_lu, on='city_id', how='left')
    .select(
        'city_id', 'city_name', 'state_code',
        'temperature_c', 'humidity_pct', 'wind_speed_kmh',
        F.trim('condition').alias('condition'), 'recorded_at'
    ).filter(F.col('city_id').isNotNull())
    .withColumn('weather_key', F.monotonically_increasing_id())
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('weather_key', 'city_id', 'city_name', 'state_code',
            'temperature_c', 'humidity_pct', 'wind_speed_kmh', 'condition', 'recorded_at',
            'gold_ingested_at', 'gold_pipeline')
)
write_dim(dim_weather, 'DimWeather')

# DimTime (generated 2020–2030)
dim_time = (
    spark.range(365 * 24 * 11)
    .select(
        F.expr("timestamp '2020-01-01 00:00:00' + make_interval(0, 0, 0, 0, cast(id as int), 0, 0)").alias('ts')
    )
    .select(
        'ts', F.to_date('ts').alias('date'),
        F.year('ts').alias('year'), F.month('ts').alias('month'),
        F.dayofmonth('ts').alias('day'), F.hour('ts').alias('hour'),
        F.date_format('ts', 'EEEE').alias('day_of_week'),
        F.dayofweek('ts').alias('day_of_week_num'),
        F.when(F.dayofweek('ts').isin(1, 7), F.lit(True)).otherwise(F.lit(False)).alias('is_weekend'),
        F.quarter('ts').alias('quarter'),
        F.when(F.month('ts') >= 7,
            F.concat(F.lit('FY'), (F.year('ts') + 1).cast('string'))
        ).otherwise(
            F.concat(F.lit('FY'), F.year('ts').cast('string'))
        ).alias('financial_year_au'),
    )
    .withColumn('time_key', F.concat_ws('',
        F.col('year').cast('string'),
        F.lpad(F.col('month').cast('string'), 2, '0'),
        F.lpad(F.col('day').cast('string'),   2, '0'),
        F.lpad(F.col('hour').cast('string'),  2, '0'),
    ))
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('time_key', 'ts', 'date', 'year', 'month', 'day', 'hour',
            'day_of_week', 'day_of_week_num', 'is_weekend', 'quarter',
            'financial_year_au', 'gold_ingested_at', 'gold_pipeline')
)
write_dim(dim_time, 'DimTime')

print('\nAll 13 dims written.')

In [ ]:
# Cell 6 — Load dim surrogate key lookups (for fact joins)
sk_station  = read_dim('DimStation').select('station_key', 'station_id')
sk_charger  = read_dim('DimCharger').filter(F.col('is_current') == True).select('charger_key', 'charger_id')
sk_customer = read_dim('DimCustomer').filter(F.col('is_current') == True).select('customer_key', 'customer_id')
sk_vehicle  = read_dim('DimVehicle').select('vehicle_key', 'vehicle_id')
sk_tariff   = read_dim('DimTariff').filter(F.col('is_current') == True).select('tariff_key', 'tariff_id')
sk_employee = read_dim('DimEmployee').select('employee_key', 'employee_id')
sk_card     = read_dim('DimChargeCard').select('card_key', 'card_id')

print('Surrogate key lookups ready')

In [ ]:
# Cell 7 — FACTS
print('=== FACTS ===')

# FactChargingSession
payments_agg = (
    payments_raw.groupBy('session_id').agg(
        F.sum('amount').alias('total_payment_aud'),
        F.count('payment_id').alias('payment_count'),
    )
)
fact_session = (
    sessions_raw
    .join(sk_station,  on='station_id',  how='left')
    .join(sk_charger,  on='charger_id',  how='left')
    .join(sk_customer, on='customer_id', how='left')
    .join(sk_vehicle,  on='vehicle_id',  how='left')
    .join(sk_tariff,   on='tariff_id',   how='left')
    .join(payments_agg, on='session_id', how='left')
    .withColumn('start_time_key', time_key_expr(F.col('start_time')))
    .withColumn('session_date',   F.to_date('start_time'))
    .withColumn('session_year',   F.year('start_time'))
    .withColumn('session_month',  F.month('start_time'))
    .withColumn('expected_cost_aud',
        F.when(F.col('price_per_kwh').isNotNull() & F.col('energy_kwh').isNotNull(),
            (F.col('price_per_kwh') * F.col('energy_kwh')
             + F.coalesce(F.col('price_per_min'), F.lit(0)) * F.col('duration_min')
            ).cast('decimal(10,2)')
        )
    )
    .withColumn('reconciliation_status',
        F.when(F.col('total_payment_aud').isNull(), F.lit('UNMATCHED'))
        .when(F.col('expected_cost_aud').isNotNull() &
              (F.abs(F.col('total_payment_aud') - F.col('expected_cost_aud')) < F.lit(0.01)),
              F.lit('MATCH'))
        .otherwise(F.lit('MISMATCH'))
    )
    .withColumn('cost_per_kwh',
        F.when(F.col('energy_kwh').isNotNull() & (F.col('energy_kwh') > 0) &
               F.col('total_payment_aud').isNotNull(),
            (F.col('total_payment_aud') / F.col('energy_kwh')).cast('decimal(8,4)')
        )
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select(
        'session_id', 'station_key', 'charger_key', 'customer_key', 'vehicle_key', 'tariff_key',
        'start_time_key', 'session_date', 'session_year', 'session_month',
        F.col('start_time').alias('session_start_ts'), F.col('end_time').alias('session_end_ts'),
        'duration_min', 'energy_kwh', F.col('price_per_kwh').alias('tariff_rate_per_kwh'),
        'expected_cost_aud', 'total_payment_aud', 'payment_count', 'cost_per_kwh',
        'reconciliation_status', F.col('status').alias('session_status'),
        'gold_ingested_at', 'gold_pipeline',
    )
)
write_fact(fact_session, 'FactChargingSession', ['session_id'], ['session_year', 'session_month'])

# FactEnergyConsumption
fact_energy = (
    sessions_raw
    .join(sk_charger, on='charger_id', how='left')
    .join(sk_station, on='station_id', how='left')
    .withColumn('session_hour_ts', F.date_trunc('hour', F.col('start_time')))
    .withColumn('time_key', time_key_expr(F.col('start_time')))
    .groupBy('charger_key', 'charger_id', 'station_key', 'station_id', 'session_hour_ts', 'time_key')
    .agg(
        F.count('session_id').alias('session_count'),
        F.sum('energy_kwh').cast('decimal(12,4)').alias('total_energy_kwh'),
        F.avg('energy_kwh').cast('decimal(10,4)').alias('avg_energy_kwh'),
        F.sum('duration_min').cast('decimal(12,2)').alias('total_duration_min'),
        F.avg('duration_min').cast('decimal(8,2)').alias('avg_duration_min'),
    )
    .withColumn('consumption_year',  F.year('session_hour_ts'))
    .withColumn('consumption_month', F.month('session_hour_ts'))
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('charger_key', 'charger_id', 'station_key', 'station_id',
            'session_hour_ts', 'time_key', 'consumption_year', 'consumption_month',
            'session_count', 'total_energy_kwh', 'avg_energy_kwh',
            'total_duration_min', 'avg_duration_min', 'gold_ingested_at', 'gold_pipeline')
)
write_fact(fact_energy, 'FactEnergyConsumption', ['charger_key', 'session_hour_ts'], ['consumption_year', 'consumption_month'])

# FactPayments
fact_payments = (
    payments_raw
    .join(sk_customer, on=payments_raw['customer_id'] == sk_customer['customer_id'], how='left').drop(sk_customer['customer_id'])
    .join(sk_card,     on=payments_raw['card_id']     == sk_card['card_id'],         how='left').drop(sk_card['card_id'])
    .withColumn('payment_time_key', time_key_expr(F.col('payment_time')))
    .withColumn('payment_year',  F.year('payment_time'))
    .withColumn('payment_month', F.month('payment_time'))
    .withColumn('gst_aud',       (F.col('amount') * F.lit(0.10)).cast('decimal(10,2)'))
    .withColumn('amount_ex_gst', (F.col('amount') * F.lit(0.9091)).cast('decimal(10,2)'))
    .withColumn('refund_flag',
        F.when(F.lower(F.col('status')).isin('refunded', 'reversed', 'refund'), F.lit(True)).otherwise(F.lit(False))
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select(
        'payment_id', 'session_id', 'customer_key', 'customer_id', 'card_key', 'card_id',
        'payment_time_key', 'payment_year', 'payment_month',
        F.col('payment_time').alias('payment_ts'),
        F.col('amount').alias('amount_aud'), 'amount_ex_gst', 'gst_aud',
        F.trim('method').alias('payment_method'), F.trim('status').alias('payment_status'),
        'refund_flag', 'gold_ingested_at', 'gold_pipeline',
    )
)
write_fact(fact_payments, 'FactPayments', ['payment_id'], ['payment_year', 'payment_month'])

# FactMaintenance
fact_maintenance = (
    maintenance_r
    .join(sk_charger, on='charger_id', how='left')
    .join(sk_station, on='station_id', how='left')
    .join(sk_employee, on=maintenance_r['assigned_employee_id'] == sk_employee['employee_id'], how='left').drop(sk_employee['employee_id'])
    .withColumn('event_time_key', time_key_expr(F.col('created_at')))
    .withColumn('event_year',  F.year('created_at'))
    .withColumn('event_month', F.month('created_at'))
    .withColumn('resolution_hours',
        F.when(F.col('resolved_at').isNotNull() & F.col('created_at').isNotNull(),
            ((F.col('resolved_at').cast('long') - F.col('created_at').cast('long')) / 3600).cast('decimal(10,2)')
        )
    )
    .withColumn('is_resolved', F.when(F.col('resolved_at').isNotNull(), F.lit(True)).otherwise(F.lit(False)))
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select(
        'event_id', 'charger_key', 'charger_id', 'station_key', 'station_id',
        F.col('employee_key').alias('assigned_employee_key'), F.col('assigned_employee_id'),
        'event_time_key', 'event_year', 'event_month',
        F.col('created_at').alias('event_created_ts'), F.col('resolved_at').alias('event_resolved_ts'),
        'resolution_hours', 'is_resolved',
        F.trim('event_type').alias('event_type'), F.trim('severity').alias('severity'),
        F.trim('status').alias('event_status'), 'gold_ingested_at', 'gold_pipeline',
    )
)
write_fact(fact_maintenance, 'FactMaintenance', ['event_id'], ['event_year', 'event_month'])

# FactDeviceTelemetry
fact_telemetry = (
    rt_sessions
    .join(sk_charger,  on='charger_id',  how='left')
    .join(sk_station,  on='station_id',  how='left')
    .join(sk_customer, on='customer_id', how='left')
    .withColumn('telemetry_time_key', time_key_expr(F.col('plug_in_ts')))
    .withColumn('telemetry_year',  F.year('plug_in_ts'))
    .withColumn('telemetry_month', F.month('plug_in_ts'))
    .withColumn('power_efficiency_pct',
        F.when(
            F.col('peak_kw').isNotNull() & (F.col('peak_kw') > 0)
            & F.col('duration_min').isNotNull() & (F.col('duration_min') > 0),
            (F.col('energy_kwh') / (F.col('peak_kw') * F.col('duration_min') / 60) * 100).cast('decimal(6,2)')
        )
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select(
        'session_id', 'charger_key', 'charger_id', 'station_key', 'station_id',
        'customer_key', 'customer_id',
        'telemetry_time_key', 'telemetry_year', 'telemetry_month',
        F.col('plug_in_ts').alias('session_start_ts'), F.col('plug_out_ts').alias('session_end_ts'),
        'duration_min', 'energy_kwh',
        F.col('peak_kw').alias('peak_power_kw'),
        F.col('raw_device_temp_c').alias('device_temp_c'),
        'signal_strength_dbm', 'firmware_ver', 'power_efficiency_pct',
        F.trim('session_status').alias('session_status'),
        'gold_ingested_at', 'gold_pipeline',
    )
)
write_fact(fact_telemetry, 'FactDeviceTelemetry', ['session_id'], ['telemetry_year', 'telemetry_month'])

# FactComplaints
fact_complaints = (
    complaints_r
    .join(sk_customer, on=complaints_r['customer_id'] == sk_customer['customer_id'], how='left').drop(sk_customer['customer_id'])
    .join(sk_station,  on=complaints_r['station_id']  == sk_station['station_id'],   how='left').drop(sk_station['station_id'])
    .withColumn('complaint_time_key', time_key_expr(F.col('created_at')))
    .withColumn('complaint_year',  F.year('created_at'))
    .withColumn('complaint_month', F.month('created_at'))
    .withColumn('resolution_days',
        F.when(
            F.col('updated_at').isNotNull() & F.col('created_at').isNotNull()
            & F.lower(F.col('status')).isin('resolved', 'closed'),
            ((F.col('updated_at').cast('long') - F.col('created_at').cast('long')) / 86400).cast('decimal(8,2)')
        )
    )
    .withColumn('is_resolved',
        F.when(F.lower(F.col('status')).isin('resolved', 'closed'), F.lit(True)).otherwise(F.lit(False))
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select(
        'complaint_id', 'customer_key', 'customer_id', 'station_key', 'station_id',
        'complaint_time_key', 'complaint_year', 'complaint_month',
        F.col('created_at').alias('complaint_created_ts'), F.col('updated_at').alias('complaint_updated_ts'),
        F.trim('category').alias('complaint_category'), F.trim('priority').alias('priority'),
        F.trim('status').alias('complaint_status'), 'is_resolved', 'resolution_days',
        'gold_ingested_at', 'gold_pipeline',
    )
)
write_fact(fact_complaints, 'FactComplaints', ['complaint_id'], ['complaint_year', 'complaint_month'])

# FactStationUtilisation
sk_station_full = read_dim('DimStation').select('station_key', 'station_id', 'total_chargers')
fact_utilisation = (
    sessions_raw
    .join(sk_station_full, on='station_id', how='left')
    .withColumn('session_hour_ts', F.date_trunc('hour', F.col('start_time')))
    .withColumn('time_key', time_key_expr(F.col('start_time')))
    .groupBy('station_key', 'station_id', 'session_hour_ts', 'time_key', 'total_chargers')
    .agg(
        F.count('session_id').alias('session_count'),
        F.sum('energy_kwh').cast('decimal(12,4)').alias('total_energy_kwh'),
        F.sum('duration_min').cast('decimal(12,2)').alias('total_active_min'),
        F.countDistinct('charger_id').alias('active_charger_count'),
    )
    .withColumn('utilisation_year',  F.year('session_hour_ts'))
    .withColumn('utilisation_month', F.month('session_hour_ts'))
    .withColumn('utilisation_pct',
        F.when(F.col('total_chargers').isNotNull() & (F.col('total_chargers') > 0),
            F.least(
                (F.col('total_active_min') / (F.col('total_chargers') * 60) * 100).cast('decimal(6,2)'),
                F.lit(100.0).cast('decimal(6,2)')
            )
        )
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select(
        'station_key', 'station_id', 'session_hour_ts', 'time_key',
        'utilisation_year', 'utilisation_month', 'total_chargers',
        'session_count', 'active_charger_count', 'total_energy_kwh',
        'total_active_min', 'utilisation_pct', 'gold_ingested_at', 'gold_pipeline',
    )
)
write_fact(fact_utilisation, 'FactStationUtilisation', ['station_key', 'session_hour_ts'], ['utilisation_year', 'utilisation_month'])

print('\nAll 7 facts written.')

In [ ]:
# Cell 8 — MARTS (always full overwrite — derived from facts)
print('=== MARTS ===')

# Load fresh fact + dim tables for mart joins
f_sess = read_fact('FactChargingSession')
f_enrg = read_fact('FactEnergyConsumption')
f_pay  = read_fact('FactPayments')
f_mnt  = read_fact('FactMaintenance')
f_tele = read_fact('FactDeviceTelemetry')
f_comp = read_fact('FactComplaints')
f_util = read_fact('FactStationUtilisation')

d_sta  = read_dim('DimStation')
d_chg  = read_dim('DimCharger').filter(F.col('is_current') == True)
d_cust = read_dim('DimCustomer').filter(F.col('is_current') == True)
d_tar  = read_dim('DimTariff').filter(F.col('is_current') == True)

# --- MartRevenueByStation ---
sta_attr = d_sta.select('station_key', 'station_name', 'state_code', 'country_name', 'total_chargers')
mart_rev_sta = (
    f_sess.groupBy('station_key', 'session_year', 'session_month').agg(
        F.count('session_id').alias('total_sessions'),
        F.sum('energy_kwh').cast('decimal(14,4)').alias('total_energy_kwh'),
        F.sum('total_payment_aud').cast('decimal(14,2)').alias('total_revenue_aud'),
        F.sum('expected_cost_aud').cast('decimal(14,2)').alias('total_expected_cost_aud'),
        F.avg('duration_min').cast('decimal(8,2)').alias('avg_duration_min'),
        F.avg('total_payment_aud').cast('decimal(10,2)').alias('avg_revenue_per_session_aud'),
        F.avg('energy_kwh').cast('decimal(10,4)').alias('avg_energy_per_session_kwh'),
        F.countDistinct('customer_key').alias('unique_customers'),
        F.sum(F.when(F.col('reconciliation_status') == 'MISMATCH', 1).otherwise(0)).alias('mismatch_count'),
    )
    .join(sta_attr, on='station_key', how='left')
    .withColumn('revenue_per_charger_aud',
        F.when(F.col('total_chargers').isNotNull() & (F.col('total_chargers') > 0),
            (F.col('total_revenue_aud') / F.col('total_chargers')).cast('decimal(12,2)')
        )
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('station_key', 'station_name', 'state_code', 'country_name',
            'session_year', 'session_month',
            'total_sessions', 'unique_customers', 'total_energy_kwh', 'avg_energy_per_session_kwh',
            'total_revenue_aud', 'total_expected_cost_aud',
            'avg_revenue_per_session_aud', 'avg_duration_min',
            'total_chargers', 'revenue_per_charger_aud', 'mismatch_count',
            'gold_ingested_at', 'gold_pipeline')
)
write_mart(mart_rev_sta, 'MartRevenueByStation', ['session_year', 'session_month'])

# --- MartRevenueByTariff ---
tar_attr = d_tar.select('tariff_key', 'tariff_name', 'rate_per_kwh', 'rate_per_min', 'currency')
mart_rev_tar = (
    f_sess.groupBy('tariff_key', 'session_year', 'session_month').agg(
        F.count('session_id').alias('total_sessions'),
        F.sum('energy_kwh').cast('decimal(14,4)').alias('total_energy_kwh'),
        F.sum('total_payment_aud').cast('decimal(14,2)').alias('total_revenue_aud'),
        F.avg('duration_min').cast('decimal(8,2)').alias('avg_duration_min'),
        F.avg('cost_per_kwh').cast('decimal(8,4)').alias('avg_cost_per_kwh'),
        F.countDistinct('customer_key').alias('unique_customers'),
        F.countDistinct('station_key').alias('stations_used'),
        F.sum(F.when(F.col('reconciliation_status') == 'MATCH', 1).otherwise(0)).alias('match_count'),
        F.sum(F.when(F.col('reconciliation_status') == 'MISMATCH', 1).otherwise(0)).alias('mismatch_count'),
    )
    .join(tar_attr, on='tariff_key', how='left')
    .withColumn('match_rate_pct',
        F.when(F.col('total_sessions') > 0,
            (F.col('match_count') / F.col('total_sessions') * 100).cast('decimal(6,2)')
        )
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('tariff_key', 'tariff_name', 'rate_per_kwh', 'rate_per_min', 'currency',
            'session_year', 'session_month',
            'total_sessions', 'unique_customers', 'stations_used',
            'total_energy_kwh', 'total_revenue_aud', 'avg_duration_min', 'avg_cost_per_kwh',
            'match_count', 'mismatch_count', 'match_rate_pct', 'gold_ingested_at', 'gold_pipeline')
)
write_mart(mart_rev_tar, 'MartRevenueByTariff', ['session_year', 'session_month'])

# --- MartCustomerLifetime ---
cust_attr = d_cust.select('customer_key', 'full_name', 'email', 'city', 'state', 'country', 'signup_date')
mart_clv = (
    f_sess.groupBy('customer_key').agg(
        F.count('session_id').alias('total_sessions'),
        F.sum('total_payment_aud').cast('decimal(14,2)').alias('total_spend_aud'),
        F.sum('energy_kwh').cast('decimal(14,4)').alias('total_energy_kwh'),
        F.avg('duration_min').cast('decimal(8,2)').alias('avg_session_min'),
        F.avg('energy_kwh').cast('decimal(10,4)').alias('avg_session_kwh'),
        F.avg('total_payment_aud').cast('decimal(10,2)').alias('avg_spend_per_session_aud'),
        F.min('session_date').alias('first_session_date'),
        F.max('session_date').alias('last_session_date'),
        F.countDistinct('station_key').alias('stations_visited'),
        F.countDistinct('charger_key').alias('chargers_used'),
        F.sum(F.when(F.col('reconciliation_status') == 'MISMATCH', 1).otherwise(0)).alias('payment_mismatches'),
    )
    .join(cust_attr, on='customer_key', how='left')
    .withColumn('days_since_first_session', F.datediff(F.current_date(), F.col('first_session_date')))
    .withColumn('days_since_last_session',  F.datediff(F.current_date(), F.col('last_session_date')))
    .withColumn('spend_per_day_aud',
        F.when(F.col('days_since_first_session').isNotNull() & (F.col('days_since_first_session') > 0),
            (F.col('total_spend_aud') / F.col('days_since_first_session')).cast('decimal(10,4)')
        )
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('customer_key', 'full_name', 'email', 'city', 'state', 'country', 'signup_date',
            'total_sessions', 'total_spend_aud', 'total_energy_kwh',
            'avg_session_min', 'avg_session_kwh', 'avg_spend_per_session_aud',
            'first_session_date', 'last_session_date',
            'days_since_first_session', 'days_since_last_session', 'spend_per_day_aud',
            'stations_visited', 'chargers_used', 'payment_mismatches',
            'gold_ingested_at', 'gold_pipeline')
)
write_mart(mart_clv, 'MartCustomerLifetime')

# --- MartMaintenanceKPI ---
sta_attr2 = d_sta.select('station_key', 'station_name', 'state_code')
mart_maint = (
    f_mnt.groupBy('station_key', 'event_year', 'event_month').agg(
        F.count('event_id').alias('total_events'),
        F.sum(F.when(F.col('is_resolved') == True, 1).otherwise(0)).alias('resolved_events'),
        F.sum(F.when(F.col('is_resolved') == False, 1).otherwise(0)).alias('unresolved_events'),
        F.avg(F.when(F.col('is_resolved') == True, F.col('resolution_hours'))).cast('decimal(8,2)').alias('avg_resolution_hours'),
        F.max('resolution_hours').cast('decimal(8,2)').alias('max_resolution_hours'),
        F.sum(F.when(F.col('severity') == 'critical', 1).otherwise(0)).alias('critical_events'),
        F.sum(F.when(F.col('severity') == 'high', 1).otherwise(0)).alias('high_events'),
        F.sum(F.when(F.col('severity') == 'medium', 1).otherwise(0)).alias('medium_events'),
        F.sum(F.when(F.col('severity') == 'low', 1).otherwise(0)).alias('low_events'),
        F.countDistinct('charger_key').alias('chargers_with_events'),
    )
    .join(sta_attr2, on='station_key', how='left')
    .withColumn('resolution_rate_pct',
        F.when(F.col('total_events') > 0,
            (F.col('resolved_events') / F.col('total_events') * 100).cast('decimal(6,2)')
        )
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('station_key', 'station_name', 'state_code', 'event_year', 'event_month',
            'total_events', 'resolved_events', 'unresolved_events',
            'resolution_rate_pct', 'avg_resolution_hours', 'max_resolution_hours',
            'critical_events', 'high_events', 'medium_events', 'low_events',
            'chargers_with_events', 'gold_ingested_at', 'gold_pipeline')
)
write_mart(mart_maint, 'MartMaintenanceKPI', ['event_year', 'event_month'])

# --- MartComplaintAnalysis ---
mart_comp = (
    f_comp.groupBy('station_key', 'station_id', 'complaint_year', 'complaint_month').agg(
        F.count('complaint_id').alias('total_complaints'),
        F.sum(F.when(F.col('is_resolved') == True, 1).otherwise(0)).alias('resolved_complaints'),
        F.avg(F.when(F.col('is_resolved') == True, F.col('resolution_days'))).cast('decimal(8,2)').alias('avg_resolution_days'),
        F.max('resolution_days').cast('decimal(8,2)').alias('max_resolution_days'),
        F.sum(F.when(F.col('priority') == 'high', 1).otherwise(0)).alias('high_priority_complaints'),
        F.sum(F.when(F.col('priority') == 'critical', 1).otherwise(0)).alias('critical_complaints'),
        F.countDistinct('customer_key').alias('unique_complainants'),
    )
    .join(sta_attr2, on='station_key', how='left')
    .withColumn('resolution_rate_pct',
        F.when(F.col('total_complaints') > 0,
            (F.col('resolved_complaints') / F.col('total_complaints') * 100).cast('decimal(6,2)')
        )
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('station_key', 'station_name', 'state_code', 'station_id',
            'complaint_year', 'complaint_month',
            'total_complaints', 'resolved_complaints',
            'resolution_rate_pct', 'avg_resolution_days', 'max_resolution_days',
            'high_priority_complaints', 'critical_complaints', 'unique_complainants',
            'gold_ingested_at', 'gold_pipeline')
)
write_mart(mart_comp, 'MartComplaintAnalysis', ['complaint_year', 'complaint_month'])

# --- MartPaymentReconciliation ---
sess_recon = (
    f_sess.groupBy('session_year', 'session_month').agg(
        F.count('session_id').alias('total_sessions'),
        F.sum('expected_cost_aud').cast('decimal(16,2)').alias('total_expected_aud'),
        F.sum('total_payment_aud').cast('decimal(16,2)').alias('total_billed_aud'),
        F.sum(F.when(F.col('reconciliation_status') == 'MATCH',    1).otherwise(0)).alias('matched_sessions'),
        F.sum(F.when(F.col('reconciliation_status') == 'MISMATCH', 1).otherwise(0)).alias('mismatch_sessions'),
        F.sum(F.when(F.col('reconciliation_status') == 'UNMATCHED',1).otherwise(0)).alias('unmatched_sessions'),
        F.sum(F.when(F.col('reconciliation_status') == 'MISMATCH',
            F.abs(F.col('total_payment_aud') - F.col('expected_cost_aud'))
        ).otherwise(F.lit(0))).cast('decimal(14,2)').alias('total_variance_aud'),
    )
)
pay_monthly = (
    f_pay.groupBy('payment_year', 'payment_month').agg(
        F.count('payment_id').alias('total_payments'),
        F.sum('amount_aud').cast('decimal(16,2)').alias('total_collected_aud'),
        F.sum('gst_aud').cast('decimal(14,2)').alias('total_gst_aud'),
        F.sum('amount_ex_gst').cast('decimal(16,2)').alias('total_ex_gst_aud'),
        F.sum(F.when(F.col('refund_flag') == True, F.col('amount_aud')).otherwise(F.lit(0))).cast('decimal(14,2)').alias('total_refunded_aud'),
        F.sum(F.when(F.col('refund_flag') == True, 1).otherwise(0)).alias('refund_count'),
    )
)
mart_recon = (
    sess_recon
    .join(pay_monthly.withColumnRenamed('payment_year', 'session_year')
                     .withColumnRenamed('payment_month', 'session_month'),
          on=['session_year', 'session_month'], how='left')
    .withColumn('match_rate_pct',
        F.when(F.col('total_sessions') > 0,
            (F.col('matched_sessions') / F.col('total_sessions') * 100).cast('decimal(6,2)')
        )
    )
    .withColumn('collection_rate_pct',
        F.when(F.col('total_expected_aud').isNotNull() & (F.col('total_expected_aud') > 0),
            (F.col('total_collected_aud') / F.col('total_expected_aud') * 100).cast('decimal(6,2)')
        )
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('session_year', 'session_month',
            'total_sessions', 'matched_sessions', 'mismatch_sessions', 'unmatched_sessions',
            'match_rate_pct',
            'total_expected_aud', 'total_billed_aud', 'total_variance_aud',
            'total_payments', 'total_collected_aud',
            'total_gst_aud', 'total_ex_gst_aud', 'collection_rate_pct',
            'refund_count', 'total_refunded_aud', 'gold_ingested_at', 'gold_pipeline')
)
write_mart(mart_recon, 'MartPaymentReconciliation', ['session_year', 'session_month'])

# --- MartStationHealthDashboard ---
sta_attr3 = d_sta.select('station_key', 'station_name', 'state_code', 'total_chargers')
util_m = (
    f_util.groupBy('station_key', 'utilisation_year', 'utilisation_month').agg(
        F.avg('utilisation_pct').cast('decimal(6,2)').alias('avg_utilisation_pct'),
        F.max('utilisation_pct').cast('decimal(6,2)').alias('peak_utilisation_pct'),
        F.sum('session_count').alias('total_sessions'),
        F.sum('total_energy_kwh').cast('decimal(14,4)').alias('total_energy_kwh'),
    )
)
maint_m = (
    f_mnt.groupBy('station_key', 'event_year', 'event_month').agg(
        F.count('event_id').alias('maintenance_events'),
        F.sum(F.when(F.col('severity').isin('critical', 'high'), 1).otherwise(0)).alias('critical_high_events'),
        F.avg('resolution_hours').cast('decimal(8,2)').alias('avg_maintenance_resolution_h'),
    )
)
comp_m = (
    f_comp.groupBy('station_key', 'complaint_year', 'complaint_month').agg(
        F.count('complaint_id').alias('complaint_count'),
        F.sum(F.when(F.col('priority').isin('critical', 'high'), 1).otherwise(0)).alias('critical_high_complaints'),
    )
)
mart_health = (
    util_m
    .join(maint_m.withColumnRenamed('event_year', 'utilisation_year')
                 .withColumnRenamed('event_month', 'utilisation_month'),
          on=['station_key', 'utilisation_year', 'utilisation_month'], how='left')
    .join(comp_m.withColumnRenamed('complaint_year', 'utilisation_year')
                .withColumnRenamed('complaint_month', 'utilisation_month'),
          on=['station_key', 'utilisation_year', 'utilisation_month'], how='left')
    .join(sta_attr3, on='station_key', how='left')
    .withColumn('maintenance_per_charger',
        F.when(F.col('total_chargers').isNotNull() & (F.col('total_chargers') > 0),
            (F.coalesce(F.col('maintenance_events'), F.lit(0)) / F.col('total_chargers')).cast('decimal(8,2)')
        )
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('station_key', 'station_name', 'state_code', 'total_chargers',
            F.col('utilisation_year').alias('report_year'),
            F.col('utilisation_month').alias('report_month'),
            'total_sessions', 'total_energy_kwh',
            'avg_utilisation_pct', 'peak_utilisation_pct',
            F.coalesce(F.col('maintenance_events'), F.lit(0)).alias('maintenance_events'),
            F.coalesce(F.col('critical_high_events'), F.lit(0)).alias('critical_high_maintenance'),
            'avg_maintenance_resolution_h', 'maintenance_per_charger',
            F.coalesce(F.col('complaint_count'), F.lit(0)).alias('complaint_count'),
            F.coalesce(F.col('critical_high_complaints'), F.lit(0)).alias('critical_high_complaints'),
            'gold_ingested_at', 'gold_pipeline')
)
write_mart(mart_health, 'MartStationHealthDashboard', ['report_year', 'report_month'])

print('\nAll 9 marts written.')

In [ ]:
# Cell 9 — Run Summary
all_dims  = ['DimCountry','DimState','DimCity','DimStation','DimCharger','DimCustomer',
             'DimVehicle','DimEmployee','DimFranchisePartner','DimChargeCard','DimTariff','DimWeather','DimTime']
all_facts = ['FactChargingSession','FactEnergyConsumption','FactPayments','FactMaintenance',
             'FactDeviceTelemetry','FactComplaints','FactStationUtilisation']
all_marts = ['MartRevenueByStation','MartRevenueByTariff','MartCustomerLifetime','MartMaintenanceKPI',
             'MartComplaintAnalysis','MartPaymentReconciliation','MartStationHealthDashboard']

print('=' * 65)
print('GOLD DIMENSIONAL v3 — RUN SUMMARY')
print('=' * 65)
print(f'LOAD_TYPE : {LOAD_TYPE}   RUN_DATE : {RUN_DATE}   RUN_TS : {RUN_TS}')
print()
print('DIMS:')
for d in all_dims:
    try:
        cnt = spark.read.format('delta').load(f'{GOLD_DIM}/{d}').count()
        print(f'  {d:<25} {cnt:>8} rows')
    except Exception as e:
        print(f'  {d:<25} ERROR: {e}')
print('\nFACTS:')
for f in all_facts:
    try:
        cnt = spark.read.format('delta').load(f'{GOLD_FACT}/{f}').count()
        print(f'  {f:<30} {cnt:>8} rows')
    except Exception as e:
        print(f'  {f:<30} ERROR: {e}')
print('\nMARTS:')
for m in all_marts:
    try:
        cnt = spark.read.format('delta').load(f'{GOLD_MART}/{m}').count()
        print(f'  {m:<35} {cnt:>8} rows')
    except Exception as e:
        print(f'  {m:<35} ERROR: {e}')
print()
print('Skipped (source data missing):')
print('  MartEnergyEfficiency    — built in 03_aggregated_marts.ipynb (standalone)')
print('  MartChargerPerformance  — built in 03_aggregated_marts.ipynb (standalone)')
print('  MartFleetUtilisation    — no telematics/fleet source data')
print('  FactFleetUtilisation    — no telematics/fleet source data')
print('  FactInvoice amounts     — invoice PDFs not parsed into Silver')